# Phase 4C - Minimal Merge Train-Val Smoke Run

This notebook builds a minimal **joint ABSA smoke run** on top of the already validated ATE and ASC exports.

Core idea:
- use **one shared encoder**;
- add one **ATE token-classification head**;
- add one **ASC sentence-level classification head**;
- run a very small joint train/val loop to verify the merged interface is feasible.

This is **not** a pipeline such as `ATE -> ASC` or `ASC -> ATE`.
It is a multitask baseline with a shared backbone and two task-specific heads.

In [1]:
import copy
import json
import math
import random
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

d:\anaconda3\envs\tpwng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Configuration

The smoke settings intentionally stay conservative and follow the same direction as Phase 4A / 4B:
- `microsoft/deberta-v3-base`
- small max length
- frozen shared encoder
- only the two task heads are trainable
- tiny SGD steps with strong finite checks

In [2]:
ROOT = Path.cwd()
PHASE2_DIR = ROOT / "outputs_phase2"
OUTPUT_DIR = ROOT / "outputs_phase4_merge_smoke"
OUTPUT_DIR.mkdir(exist_ok=True)

ATE_TRAIN_PATH = PHASE2_DIR / "ate_train.jsonl"
ATE_VAL_PATH = PHASE2_DIR / "ate_val.jsonl"
ASC_TRAIN_PATH = PHASE2_DIR / "asc_train.jsonl"
ASC_VAL_PATH = PHASE2_DIR / "asc_val.jsonl"

BACKBONE_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 128
ATE_BATCH_SIZE = 4
ASC_BATCH_SIZE = 8
LR = 1e-6
WEIGHT_DECAY = 0.0
FREEZE_BACKBONE_FOR_SMOKE = True
MAX_TRAIN_STEPS = 20
MAX_VAL_BATCHES = 5
IGNORE_INDEX = -100

ASC_LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ASC_ID2LABEL = {v: k for k, v in ASC_LABEL2ID.items()}
ATE_LABEL2ID = {"O": 0, "B-ASP": 1, "I-ASP": 2}
ATE_ID2LABEL = {v: k for k, v in ATE_LABEL2ID.items()}

REPORT_PATH = OUTPUT_DIR / "merge_min_train_val_smoke_report.json"
CHECKLIST_PATH = OUTPUT_DIR / "phase4c_checklist.csv"

pprint({
    "backbone": BACKBONE_NAME,
    "max_len": MAX_LEN,
    "ate_batch_size": ATE_BATCH_SIZE,
    "asc_batch_size": ASC_BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "freeze_backbone_for_smoke": FREEZE_BACKBONE_FOR_SMOKE,
    "max_train_steps": MAX_TRAIN_STEPS,
    "max_val_batches": MAX_VAL_BATCHES,
})

{'asc_batch_size': 8,
 'ate_batch_size': 4,
 'backbone': 'microsoft/deberta-v3-base',
 'freeze_backbone_for_smoke': True,
 'lr': 1e-06,
 'max_len': 128,
 'max_train_steps': 20,
 'max_val_batches': 5,
 'weight_decay': 0.0}


## Load Phase 2 Exports

We directly consume the task-specific files exported by Phase 2:
- ATE uses `tokens` + `bio_labels`
- ASC uses `text` + `aspect` + `label`

This keeps Phase 4C aligned with the existing single-task baselines.

In [3]:
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

ate_train = load_jsonl(ATE_TRAIN_PATH)
ate_val = load_jsonl(ATE_VAL_PATH)
asc_train = load_jsonl(ASC_TRAIN_PATH)
asc_val = load_jsonl(ASC_VAL_PATH)

print("ATE train/val rows:", len(ate_train), len(ate_val))
print("ASC train/val rows:", len(asc_train), len(asc_val))

print("ATE sample keys:", sorted(ate_train[0].keys()))
print("ASC sample keys:", sorted(asc_train[0].keys()))

ATE train/val rows: 8509 1063
ASC train/val rows: 11369 1417
ATE sample keys: ['aspect_alignment_audit', 'aspects', 'bio_labels', 'bio_span_count_mismatch', 'doc_id', 'has_all_o_but_matched_aspect', 'num_bio_recovered_spans', 'num_full_coverage_aspects', 'num_matched_aspects', 'num_partial_coverage_aspects', 'num_zero_coverage_aspects', 'raw_id', 'split', 'text', 'token_offsets', 'tokens']
ASC sample keys: ['aspect', 'char_from', 'char_to', 'doc_id', 'input_text_template', 'match_type', 'raw_id', 'sentence', 'sentiment', 'split']


## Tokenization And Batch Building

The ATE path follows the same alignment logic as Phase 4B:
- word-level BIO labels are expanded to subwords via `word_ids()`
- only the first subword keeps the word label
- the rest are masked with `IGNORE_INDEX`

The ASC path follows the same setup as Phase 4A:
- encode `(text, aspect)` as a pair
- predict a single sentiment label

In [4]:
tokenizer = AutoTokenizer.from_pretrained(BACKBONE_NAME)
print("Loaded tokenizer:", BACKBONE_NAME)

class JsonlListDataset(Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        return self.records[idx]

def encode_asc_example(example):
    sentence = example.get("sentence", example.get("text"))
    aspect = example["aspect"]
    sentiment = example.get("sentiment", example.get("label"))
    encoded = tokenizer(
        sentence,
        aspect,
        truncation=True,
        max_length=MAX_LEN,
        return_attention_mask=True,
    )
    encoded["labels"] = ASC_LABEL2ID[sentiment]
    return encoded

def collate_asc_batch(batch):
    encodings = [encode_asc_example(ex) for ex in batch]
    labels = torch.tensor([x.pop("labels") for x in encodings], dtype=torch.long)
    padded = tokenizer.pad(encodings, padding=True, return_tensors="pt")
    padded["labels"] = labels
    return padded

def encode_ate_example(example):
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN,
        return_attention_mask=True,
    )
    word_ids = tokenized.word_ids()
    labels = []
    prev_word_id = None
    for word_id in word_ids:
        if word_id is None:
            labels.append(IGNORE_INDEX)
        elif word_id != prev_word_id:
            labels.append(ATE_LABEL2ID[example["bio_labels"][word_id]])
        else:
            labels.append(IGNORE_INDEX)
        prev_word_id = word_id
    encoded = {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "labels": labels,
    }
    return encoded

def collate_ate_batch(batch):
    encodings = [encode_ate_example(ex) for ex in batch]
    label_lists = [x.pop("labels") for x in encodings]
    padded = tokenizer.pad(encodings, padding=True, return_tensors="pt")
    max_seq_len = padded["input_ids"].shape[1]
    padded_labels = []
    for labels in label_lists:
        pad_len = max_seq_len - len(labels)
        padded_labels.append(labels + [IGNORE_INDEX] * pad_len)
    padded["labels"] = torch.tensor(padded_labels, dtype=torch.long)
    return padded

ate_train_ds = JsonlListDataset(ate_train)
ate_val_ds = JsonlListDataset(ate_val)
asc_train_ds = JsonlListDataset(asc_train)
asc_val_ds = JsonlListDataset(asc_val)

ate_train_loader = DataLoader(ate_train_ds, batch_size=ATE_BATCH_SIZE, shuffle=True, collate_fn=collate_ate_batch)
ate_val_loader = DataLoader(ate_val_ds, batch_size=ATE_BATCH_SIZE, shuffle=False, collate_fn=collate_ate_batch)
asc_train_loader = DataLoader(asc_train_ds, batch_size=ASC_BATCH_SIZE, shuffle=True, collate_fn=collate_asc_batch)
asc_val_loader = DataLoader(asc_val_ds, batch_size=ASC_BATCH_SIZE, shuffle=False, collate_fn=collate_asc_batch)

ate_batch = next(iter(ate_train_loader))
asc_batch = next(iter(asc_train_loader))
print("ATE batch shapes:", {k: tuple(v.shape) for k, v in ate_batch.items()})
print("ASC batch shapes:", {k: tuple(v.shape) for k, v in asc_batch.items()})

Loaded tokenizer: microsoft/deberta-v3-base
ATE batch shapes: {'input_ids': (4, 20), 'attention_mask': (4, 20), 'labels': (4, 20)}
ASC batch shapes: {'input_ids': (8, 20), 'token_type_ids': (8, 20), 'attention_mask': (8, 20), 'labels': (8,)}


## Shared Encoder Joint Model

The merged baseline below follows the usual multitask ABSA intuition:
- one encoder builds contextual representations;
- the ATE head predicts token-level BIO labels;
- the ASC head predicts sentence-level sentiment for a `(text, aspect)` pair.

For this smoke run we freeze the encoder and train only the two heads, because the goal is to verify the merged training path first.

In [5]:
class JointABSASmokeModel(nn.Module):
    def __init__(self, backbone_name, num_ate_labels, num_asc_labels, freeze_backbone=True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(backbone_name)
        hidden_size = self.encoder.config.hidden_size
        dropout_prob = getattr(self.encoder.config, "hidden_dropout_prob", 0.1)
        self.dropout = nn.Dropout(dropout_prob)
        self.ate_classifier = nn.Linear(hidden_size, num_ate_labels)
        self.asc_classifier = nn.Linear(hidden_size, num_asc_labels)
        if freeze_backbone:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def forward_ate(self, input_ids, attention_mask, labels=None, token_type_ids=None):
        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids
        outputs = self.encoder(**encoder_kwargs)
        token_hidden = self.dropout(outputs.last_hidden_state)
        token_hidden = token_hidden.to(self.ate_classifier.weight.dtype)
        logits = self.ate_classifier(token_hidden)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return {"loss": loss, "logits": logits}

    def forward_asc(self, input_ids, attention_mask, labels=None, token_type_ids=None):
        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids
        outputs = self.encoder(**encoder_kwargs)
        cls_hidden = self.dropout(outputs.last_hidden_state[:, 0, :])
        cls_hidden = cls_hidden.to(self.asc_classifier.weight.dtype)
        logits = self.asc_classifier(cls_hidden)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
        return {"loss": loss, "logits": logits}

model = JointABSASmokeModel(
    backbone_name=BACKBONE_NAME,
    num_ate_labels=len(ATE_LABEL2ID),
    num_asc_labels=len(ASC_LABEL2ID),
    freeze_backbone=FREEZE_BACKBONE_FOR_SMOKE,
).to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print("Model loaded:", BACKBONE_NAME)
print("Freeze backbone for smoke:", FREEZE_BACKBONE_FOR_SMOKE)
print("Trainable params:", trainable_params)
print("Total params:", total_params)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 49488.84it/s]
DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect

Model loaded: microsoft/deberta-v3-base
Freeze backbone for smoke: True
Trainable params: 4614
Total params: 183836166


## Single-Batch Forward And Backward Check

Before a joint train loop, we first verify:
- one ATE batch can complete forward/backward;
- one ASC batch can complete forward/backward;
- both losses are finite.

In [6]:
ate_batch_device = {k: v.to(DEVICE) for k, v in ate_batch.items()}
asc_batch_device = {k: v.to(DEVICE) for k, v in asc_batch.items()}

model.zero_grad(set_to_none=True)
ate_out = model.forward_ate(**ate_batch_device)
asc_out = model.forward_asc(**asc_batch_device)

print("ATE finite loss:", bool(torch.isfinite(ate_out["loss"])))
print("ASC finite loss:", bool(torch.isfinite(asc_out["loss"])))

joint_debug_loss = 0.5 * (ate_out["loss"] + asc_out["loss"])
joint_debug_loss.backward()
model.zero_grad(set_to_none=True)
print("One joint backward pass completed successfully.")

ATE finite loss: True
ASC finite loss: True
One joint backward pass completed successfully.


## Minimal Joint Train / Val Smoke Run

Joint training strategy for the smoke run:
- pull one batch from ATE and one batch from ASC;
- compute both task losses;
- average them into one joint loss;
- update only the task heads;
- if a non-finite value appears, stop early and restore the last safe model state.

This is intentionally simple. The goal is not to optimize performance yet, but to prove the merged training loop is stable enough to run.

In [7]:
model = JointABSASmokeModel(
    backbone_name=BACKBONE_NAME,
    num_ate_labels=len(ATE_LABEL2ID),
    num_asc_labels=len(ASC_LABEL2ID),
    freeze_backbone=FREEZE_BACKBONE_FOR_SMOKE,
).to(DEVICE)

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
)

train_steps = 0
ate_val_batches_seen = 0
asc_val_batches_seen = 0
joint_train_losses = []
ate_val_losses = []
asc_val_losses = []
ate_val_masked_acc = []
asc_val_acc = []
last_finite_state = copy.deepcopy(model.state_dict())

model.train()
for ate_train_batch, asc_train_batch in zip(ate_train_loader, asc_train_loader):
    if train_steps >= MAX_TRAIN_STEPS:
        break
    step_start_state = copy.deepcopy(model.state_dict())
    ate_train_batch = {k: v.to(DEVICE) for k, v in ate_train_batch.items()}
    asc_train_batch = {k: v.to(DEVICE) for k, v in asc_train_batch.items()}
    optimizer.zero_grad()

    ate_out = model.forward_ate(**ate_train_batch)
    asc_out = model.forward_asc(**asc_train_batch)

    finite_ok = True
    if not torch.isfinite(ate_out["loss"]):
        print(f"Train step {train_steps + 1:02d} | ATE non-finite loss encountered, stopping early")
        finite_ok = False
    if not torch.isfinite(asc_out["loss"]):
        print(f"Train step {train_steps + 1:02d} | ASC non-finite loss encountered, stopping early")
        finite_ok = False
    if not torch.isfinite(ate_out["logits"]).all():
        print(f"Train step {train_steps + 1:02d} | ATE non-finite logits encountered, stopping early")
        finite_ok = False
    if not torch.isfinite(asc_out["logits"]).all():
        print(f"Train step {train_steps + 1:02d} | ASC non-finite logits encountered, stopping early")
        finite_ok = False
    if not finite_ok:
        model.load_state_dict(last_finite_state)
        break

    joint_loss = 0.5 * (ate_out["loss"] + asc_out["loss"])
    if not torch.isfinite(joint_loss):
        print(f"Train step {train_steps + 1:02d} | joint loss is non-finite, stopping early")
        model.load_state_dict(last_finite_state)
        break

    joint_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    trainable_params_finite = all(
        torch.isfinite(param).all().item()
        for param in model.parameters()
        if param.requires_grad
    )
    if not trainable_params_finite:
        print(f"Train step {train_steps + 1:02d} | parameters became non-finite after update, reverting and stopping early.")
        model.load_state_dict(step_start_state)
        break

    train_steps += 1
    joint_train_losses.append(joint_loss.item())
    last_finite_state = copy.deepcopy(model.state_dict())
    print(f"Train step {train_steps:02d} | joint_loss = {joint_loss.item():.6f} | ate_loss = {ate_out['loss'].item():.6f} | asc_loss = {asc_out['loss'].item():.6f}")

model.load_state_dict(last_finite_state)
model.eval()

with torch.no_grad():
    for batch in ate_val_loader:
        if ate_val_batches_seen >= MAX_VAL_BATCHES:
            break
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model.forward_ate(**batch)
        if (not torch.isfinite(out["loss"])) or (not torch.isfinite(out["logits"]).all()):
            print(f"ATE val batch {ate_val_batches_seen + 1:02d} | non-finite value encountered, stopping early")
            break
        preds = out["logits"].argmax(dim=-1)
        mask = batch["labels"] != IGNORE_INDEX
        correct = (preds[mask] == batch["labels"][mask]).sum().item()
        total = mask.sum().item()
        ate_val_losses.append(out["loss"].item())
        ate_val_masked_acc.append(correct / total if total else 0.0)
        ate_val_batches_seen += 1
        print(f"ATE val batch {ate_val_batches_seen:02d} | loss = {out['loss'].item():.6f} | masked_acc = {ate_val_masked_acc[-1]:.4f}")

with torch.no_grad():
    for batch in asc_val_loader:
        if asc_val_batches_seen >= MAX_VAL_BATCHES:
            break
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model.forward_asc(**batch)
        if (not torch.isfinite(out["loss"])) or (not torch.isfinite(out["logits"]).all()):
            print(f"ASC val batch {asc_val_batches_seen + 1:02d} | non-finite value encountered, stopping early")
            break
        preds = out["logits"].argmax(dim=-1)
        acc = (preds == batch["labels"]).float().mean().item()
        asc_val_losses.append(out["loss"].item())
        asc_val_acc.append(acc)
        asc_val_batches_seen += 1
        print(f"ASC val batch {asc_val_batches_seen:02d} | loss = {out['loss'].item():.6f} | acc = {acc:.4f}")

print("Train steps completed:", train_steps)
print("ATE val batches completed:", ate_val_batches_seen)
print("ASC val batches completed:", asc_val_batches_seen)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 39377.53it/s]
DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect

Train step 01 | joint_loss = 1.060634 | ate_loss = 1.009768 | asc_loss = 1.111499
Train step 02 | joint_loss = 1.164937 | ate_loss = 1.162767 | asc_loss = 1.167108
Train step 03 | joint_loss = 1.168298 | ate_loss = 1.169376 | asc_loss = 1.167221
Train step 04 | joint_loss = 1.020086 | ate_loss = 0.968520 | asc_loss = 1.071651
Train step 05 | joint_loss = 1.166625 | ate_loss = 1.236154 | asc_loss = 1.097096
Train step 06 | joint_loss = 1.185090 | ate_loss = 1.155346 | asc_loss = 1.214833
Train step 07 | joint_loss = 1.144520 | ate_loss = 1.126675 | asc_loss = 1.162365
Train step 08 | joint_loss = 1.031625 | ate_loss = 1.018089 | asc_loss = 1.045161
Train step 09 | joint_loss = 1.096642 | ate_loss = 1.090571 | asc_loss = 1.102712
Train step 10 | joint_loss = 1.147531 | ate_loss = 1.189102 | asc_loss = 1.105960
Train step 11 | joint_loss = 1.168049 | ate_loss = 1.187255 | asc_loss = 1.148843
Train step 12 | joint_loss = 1.157851 | ate_loss = 1.200172 | asc_loss = 1.115531
Train step 13 | 

## Save Smoke Report

The final report is intentionally compact. It is meant to answer one question:

**Can a shared-encoder joint ABSA baseline complete a minimal train/val smoke cycle without breaking the interfaces?**

In [8]:
def mean_or_none(values):
    return float(np.mean(values)) if values else None

smoke_report = {
    "backbone": BACKBONE_NAME,
    "device": str(DEVICE),
    "seed": SEED,
    "ate_train_rows": len(ate_train),
    "ate_val_rows": len(ate_val),
    "asc_train_rows": len(asc_train),
    "asc_val_rows": len(asc_val),
    "ate_batch_size": ATE_BATCH_SIZE,
    "asc_batch_size": ASC_BATCH_SIZE,
    "max_len": MAX_LEN,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "freeze_backbone_for_smoke": FREEZE_BACKBONE_FOR_SMOKE,
    "max_train_steps": MAX_TRAIN_STEPS,
    "max_val_batches": MAX_VAL_BATCHES,
    "train_steps_completed": train_steps,
    "ate_val_batches_completed": ate_val_batches_seen,
    "asc_val_batches_completed": asc_val_batches_seen,
    "mean_joint_train_loss": mean_or_none(joint_train_losses),
    "mean_ate_val_loss": mean_or_none(ate_val_losses),
    "mean_asc_val_loss": mean_or_none(asc_val_losses),
    "mean_ate_val_masked_acc": mean_or_none(ate_val_masked_acc),
    "mean_asc_val_acc": mean_or_none(asc_val_acc),
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(smoke_report, f, ensure_ascii=False, indent=2)

checklist = [
    ("ATE train file loaded", len(ate_train) > 0),
    ("ASC train file loaded", len(asc_train) > 0),
    ("Tokenizer loaded", tokenizer is not None),
    ("Joint model created", model is not None),
    ("ATE batch built", ate_batch["input_ids"].shape[0] > 0),
    ("ASC batch built", asc_batch["input_ids"].shape[0] > 0),
    ("At least one joint train step completed", train_steps > 0),
    ("At least one ATE val batch completed", ate_val_batches_seen > 0),
    ("At least one ASC val batch completed", asc_val_batches_seen > 0),
    ("Smoke report saved", REPORT_PATH.exists()),
]

check_df = pd.DataFrame(checklist, columns=["check_item", "passed"])
check_df.to_csv(CHECKLIST_PATH, index=False, encoding="utf-8-sig")

display(check_df)
pprint(smoke_report)
print("PHASE 4C MINIMAL MERGE SMOKE RUN COMPLETE:", check_df["passed"].all())
print("Report saved to:", REPORT_PATH)
print("Checklist saved to:", CHECKLIST_PATH)

,check_item,passed
0,ATE train file loaded,True
1,ASC train file loaded,True
2,Tokenizer loaded,True
3,Joint model created,True
4,ATE batch built,True
5,ASC batch built,True
6,At least one joint train step completed,True
7,At least one ATE val batch completed,True
8,At least one ASC val batch completed,True
9,Smoke report saved,True


{'asc_batch_size': 8,
 'asc_train_rows': 11369,
 'asc_val_batches_completed': 5,
 'asc_val_rows': 1417,
 'ate_batch_size': 4,
 'ate_train_rows': 8509,
 'ate_val_batches_completed': 5,
 'ate_val_rows': 1063,
 'backbone': 'microsoft/deberta-v3-base',
 'device': 'cuda',
 'freeze_backbone_for_smoke': True,
 'lr': 1e-06,
 'max_len': 128,
 'max_train_steps': 20,
 'max_val_batches': 5,
 'mean_asc_val_acc': 0.0,
 'mean_asc_val_loss': 1.1266925811767579,
 'mean_ate_val_loss': 1.0448099732398988,
 'mean_ate_val_masked_acc': 0.47314189189189193,
 'mean_joint_train_loss': 1.1193779826164245,
 'seed': 42,
 'train_steps_completed': 20,
 'weight_decay': 0.0}
PHASE 4C MINIMAL MERGE SMOKE RUN COMPLETE: True
Report saved to: d:\GitHub\NLP_PROJECT_ABSA\outputs_phase4_merge_smoke\merge_min_train_val_smoke_report.json
Checklist saved to: d:\GitHub\NLP_PROJECT_ABSA\outputs_phase4_merge_smoke\phase4c_checklist.csv
